# How to sample the posterior given iid observations (for NLE and NRE)

> Important: This how-to guide is only relevant if you have iid observations **and** you are using NLE or NRE. You can use NPE with iid observations directly by passing `sample_with="mcmc"`, `"importance"`, `"rejection"` or `"vi"` to `build_posterior` (it warns, because NPE is mostly used with one observation). If you want to use the default direct sampling (`sample_with="direct"`) for NPE with iid data, you still need to construct a permutation-invariant embedding net, which is explained in [this how-to guide](https://sbi.readthedocs.io/en/latest/how_to_guide/08_permutation_invariant_embeddings.html).

In many cases, you want to estimate a parameter set given _multiple_ observations. NLE and NRE can naturally deal with this scenario. Both of these methods can be trained on _single_ observations (i.e., just one simulation per parameter set), and can then be used to sample the posterior given an arbitrary number of observations. In `sbi`, this can be done as shown below:

```python
# Generate training dataset.
theta = prior.sample((100,))  # Example shape: (100, 3)
x = simulate(theta)  # Example shape: (100, 4)

# Train NLE or NRE network.
trainer = NLE(prior=prior)
trainer.append_simulations(theta, x).train()
posterior = trainer.build_posterior()

x_o_iid = torch.ones((20, 4))  # 20 iid observations.
samples = posterior.sample((1_000,), x=x_o_iid)
```

## Which posteriors take several observations

`x` of shape `(num_observations, *x_shape)` holds several observations. Most posteriors combine them as iid trials of one parameter set. Methods that take one observation raise an error that names the alternatives.

| Posterior | `sample()`, `log_prob()`, `potential()`, `map()` with several observations | One posterior per observation |
|---|---|---|
| MCMC, importance, rejection | iid trials | `sample_batched()` (MCMC) |
| VI after `train(x)` | iid trials: `train(x)` fits q to all observations in `x` | |
| VI after `train_amortized()` | `sample()` and `log_prob()` treat each observation separately; `potential()` and `map()` raise | `sample()`, `log_prob()`, `sample_batched()` |
| Direct (NPE default) | raises | `sample_batched()`, `log_prob_batched()` |
| Vector field (FMPE, NPSE) | iid trials (`log_prob()` up to a constant); `map()` and `leakage_correction()` raise | `sample_batched()` |
| Ensemble | as its components; with NPE components, `potential()`, the joint `map()` and `log_prob()` raise, while `sample()` and the per-component `map(individually=True)` and `log_prob(individually=True)` work | as its components |

NPE posteriors warn when they combine several observations. NLE, NRE and vector field posteriors combine them silently, because they are often used with iid data.

## Example and further explanation

For more details, see [this tutorial](https://sbi.readthedocs.io/en/latest/advanced_tutorials/12_iid_data_and_permutation_invariant_embeddings.html).